In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import matplotlib.lines as mlines
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py

from tqdm import tqdm
from glob import glob

#Specific Libraries
from matplotlib.lines import Line2D

In [ ]:
def GetPlottingDirectory(plotFileName, plotType):
    plottingDirectory = mainCodeDirectory=os.path.join(mainDirectory,"Code","PLOTTING")
    
    specificPlottingDirectory = os.path.join(plottingDirectory, plotType, 
                                             f"Simulation_{ModelData.simulationNumber}_{ModelData.res}_{ModelData.t_res}_{ModelData.Nz_str}nz")
    os.makedirs(specificPlottingDirectory, exist_ok=True)

    plottingFileName=os.path.join(specificPlottingDirectory, plotFileName)

    return plottingFileName

def SaveFigure(fig,plotType, fileName):
    plotFileName = f"{fileName}_{ModelData.res}_{ModelData.t_res}_{ModelData.Np_str}.jpg"
    plottingFileName = GetPlottingDirectory(plotFileName, plotType)
    print(f"Saving figure to {plottingFileName}")
    fig.savefig(plottingFileName, dpi=300, bbox_inches='tight')

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=1)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="Tracking_Algorithms", dataName="Lagrangian_UpdraftTracking",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import *

#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","2_Tracking_Algorithms"))
from CLASSES_TrackingAlgorithms import TrackingAlgorithms_DataLoading_Class, Results_InputOutput_Class, TrackedParcel_Loading_Class

# IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","3_Tracked_Profiles"))
from CLASSES_TrackedProfiles import TrackedProfiles_Plotting_CLASS 

import sys
dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
path=os.path.join(dir2,'Code/CodeFiles/Functions')
sys.path.append(path)

import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

sys.path.append(os.path.join(mainDirectory, "Code", "CodeFiles", "Functions", "Classes"))
from DataPlotting_Classes import DataPlotting_Classes
UltimatePlotting = DataPlotting_Classes.UltimatePlotting_Class

#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
#############################################
#LOADING DATA

In [ ]:
#READING BACK IN SUBSETTED TRACKED PARCEL DATA
Apply_FilterOutPriorAscent=False
# Apply_FilterOutPriorAscent=True

suffix = "_FilterOutPriorAscent" if Apply_FilterOutPriorAscent else ""
trackedArrays,LevelsDictionary = TrackedParcel_Loading_Class.LoadingSubsetParcelData(ModelData,DataManager,
                                                                                     Results_InputOutput_Class,
                                                                                     Apply_FilterOutPriorAscent=Apply_FilterOutPriorAscent)

In [ ]:
#############################################
#CALCULATION FUNCTIONS

In [ ]:
#Dictionary Setup Functions

def InitializeDictionary(parcelTypes,parcelDepths,
                         variableKeys=['left_right','left','right']):
    Dictionary = {}
    for parcelType in parcelTypes:
        Dictionary[parcelType] = {}
        for parcelDepth in parcelDepths:
            Dictionary[parcelType][parcelDepth] = {}
            for variableKey in variableKeys:
                Dictionary[parcelType][parcelDepth][variableKey] = {}
    return Dictionary

def GetInitialTimes(Dictionary,
                    parcelType,parcelDepth):
    #get parcel array
    parcelArray = trackedArrays[parcelType][parcelDepth]

    #get initial times
    initialTimes = parcelArray[:,1]

    #get masks
    maskLeft = parcelArray[:,4]==-1; maskRight = parcelArray[:,4]==1

    #apply left_right mask to time
    initialTimes_left_right = initialTimes[maskLeft|maskRight]
    initialTimes_left = initialTimes[maskLeft]
    initialTimes_right = initialTimes[maskRight]

    #append to dictionary
    Dictionary[parcelType][parcelDepth]['left_right'] = initialTimes_left_right
    Dictionary[parcelType][parcelDepth]['left'] = initialTimes_left
    Dictionary[parcelType][parcelDepth]['right'] = initialTimes_right
    return Dictionary

def SetupInitialTimesDictionary(parcelTypes,parcelDepths):
    initialTimesDictionary = InitializeDictionary(parcelTypes,parcelDepths)
    for parcelType in parcelTypes:
        for parcelDepth in parcelDepths:
            initialTimesDictionary = GetInitialTimes(initialTimesDictionary,parcelType,parcelDepth)
    return initialTimesDictionary

def SetupDistributionDictionary(parcelTypes, parcelDepths,
                                variableKeys=['left_right','left','right']):
    distributionDictionary = InitializeDictionary(parcelTypes, parcelDepths, variableKeys)
    for parcelType in parcelTypes:
        for parcelDepth in parcelDepths:
            for variableKey in variableKeys:
                distributionDictionary[parcelType][parcelDepth][variableKey] = np.zeros(ModelData.Ntime, dtype=int)
    return distributionDictionary

In [ ]:
#Calculation Functions

def AddAt_TimeSeriesDictionary(array,outputArray):
    np.add.at(outputArray, array[:,1], 1)
    return outputArray

def CalculateTimeSeriesDictionary(trackedArrays):
    parcelTypes=list(trackedArrays.keys());parcelDepths=list(trackedArrays['CL'].keys())
    
    outputDictionary = {}    
    for parcelType in parcelTypes:
        outputDictionary[parcelType] = {}
        for parcelDepth in parcelDepths:
            array = trackedArrays[parcelType][parcelDepth]
            outputArray = AddAt_TimeSeriesDictionary(array,np.zeros(ModelData.Ntime))
            outputDictionary[parcelType][parcelDepth] = outputArray
    return outputDictionary

def AddAt(output,index,input):
    np.add.at(output,index,input)
    return output

def CalculateDistributions(initialTimesDictionary,distributionDictionary):
    for parcelType in distributionDictionary:
        for parcelDepth in distributionDictionary[parcelType]:
            for variableKey in distributionDictionary[parcelType][parcelDepth]:            
                initialTimes = initialTimesDictionary[parcelType][parcelDepth][variableKey]
                distribution = distributionDictionary[parcelType][parcelDepth][variableKey]
                distribution = AddAt(distribution,initialTimes,1)
    return distributionDictionary

def ComputeDepthProbabilityDictionary(distributionDictionary):
    
    depthProbabilityDictionary = InitializeDictionary(parcelTypes, parcelDepths)

    for parcelType in distributionDictionary:
        for variableKey in distributionDictionary[parcelType]['SHALLOW']: #keys are the same for shallow and deep

            shallow = distributionDictionary[parcelType]['SHALLOW'][variableKey]
            deep = distributionDictionary[parcelType]['DEEP'][variableKey]

            total = shallow + deep

            # initialize arrays
            p_shallow = np.zeros_like(shallow, dtype=float)
            p_deep = np.zeros_like(deep, dtype=float)

            mask = total > 0
            p_shallow[mask] = shallow[mask] / total[mask]
            p_deep[mask] = deep[mask] / total[mask]

            # assign back into same structure
            depthProbabilityDictionary[parcelType]['SHALLOW'][variableKey] = p_shallow
            depthProbabilityDictionary[parcelType]['DEEP'][variableKey] = p_deep

    return depthProbabilityDictionary

In [ ]:
#############################################
#PLOTTING FUNCTIONS

In [ ]:
#Plotting Functions


def MakeParcelTimeSeriesFigure(timeseriesDictionary, normalize=True, fontScale=1.6,
                               figSize=None, tickSize=18):
    UP = UltimatePlotting
    [fs] = UP.GetFontStyles(fontScale)
    time = np.arange(ModelData.Ntime) * ModelData.dt / 3600 + 6

    rows = ['all', 'shallow+deep'] if normalize else ['all', 'shallow', 'deep']
    columns = [['CL', 'nonCL'], ['SBF', 'ColdPool']]
    nRows = len(rows)

    [fig, axes, _] = UP.MakeAlignedGridFigure(
        nRows=nRows, nCols=2, figSize=figSize or (11, 4 * nRows),
        colGapRatio=0.07, rowGapRatio=0.07, shareX=True, shareY=True)

    xTicks = np.arange(10, 17 + 1e-6, 1)
    linestyles = ['-', '--', '-.', ':']
    letters = "ABCDEFGH"

    for r, plotType in enumerate(rows):
        rowData = []
        for c, parcelTypes in enumerate(columns):
            rowData += PlotParcelTimeSeries(axes[r, c], timeseriesDictionary, parcelTypes,
                                            plotType, time, fontScale, normalize)
            axes[r, c].set_xlim(10, 17); axes[r, c].set_xticks(xTicks)

        # one set of y ticks per row (y is shared across the row)
        [yTicks, yLim, _] = UP.GenerateTicksAndLim(rowData + [np.array([0.0])], nTicks=6)
        axes[r, 0].set_yticks(yTicks); axes[r, 0].set_ylim(0, yLim[1])
        axes[r, 0].set_ylabel('count / max(count) (%)' if normalize else 'count')

        for c in range(2):
            if r == nRows - 1:
                axes[r, c].set_xlabel('Local time (LT)')     # bottom row only
            UP.ApplyFontStyles(axes[r, c], fs)
            axes[r, c].text(0.02, 0.96, f"{letters[2 * r + c]})", transform=axes[r, c].transAxes,
                            ha='left', va='top', fontsize=fs["title"]["fontsize"])

    # parcel-type legend (linestyles) on the top row only
    for c, parcelTypes in enumerate(columns):
        handles = [Line2D([], [], color='black', linestyle=linestyles[i % 4], label=p)
                   for i, p in enumerate(parcelTypes)]
        axes[0, c].legend(handles=handles, loc='upper right', fontsize=fs["legend"]["fontsize"])

    # color legend above the grid
    top = axes[0, 0].get_position().y1
    xc = 0.5 * (axes[0, 0].get_position().x0 + axes[0, 1].get_position().x1)
    colorHandles = [Line2D([], [], color=c, label=l) for l, c in
                    [('ALL', 'black'), ('SHALLOW', 'green'), ('DEEP', 'blue')]]
    fig.legend(handles=colorHandles, loc='lower center', ncol=3, frameon=False,
               bbox_to_anchor=(xc, top - 0.005), fontsize=fs["legend"]["fontsize"] * 1.3,
               handlelength=2.5, columnspacing=2.0)

    if tickSize is not None:
        for ax in axes.ravel():
            ax.tick_params(axis='both', which='major', labelsize=tickSize)
    return fig

def MovingAverage(y, window_mins=15, applySmoothing=True, mode='mean'):
    if not applySmoothing or not window_mins:
        return y
    window = int(window_mins * 60 / ModelData.dt)
    kernel = np.ones(window) if mode == 'sum' else np.ones(window) / window
    return np.convolve(y, kernel, mode='same')

PLOT_TYPES = {'all': [('ALL', 'black')],
              'shallow': [('SHALLOW', 'green')],
              'deep': [('DEEP', 'blue')],
              'shallow+deep': [('SHALLOW', 'green'), ('DEEP', 'blue')]}

def PlotParcelTimeSeries(ax, outputDictionary, parcelTypes, plotType, time, fontScale,
                         normalize=True):
    """Plots each parcel type (one linestyle each) and returns the plotted arrays."""
    linestyles = ['-', '--', '-.', ':']
    plotted = []
    for i, parcelType in enumerate(parcelTypes):
        for key, color in PLOT_TYPES[plotType]:
            y = MovingAverage(np.asarray(outputDictionary[parcelType][key], float),
                              mode='mean' if normalize else 'sum')
            if normalize:
                y = y / np.max(y) * 100
            UltimatePlotting.PlotLine(ax=ax, xData=time, yData=y, color=color,
                                      linestyle=linestyles[i % len(linestyles)],
                                      showLegend=False, fontScale=fontScale)
            plotted.append(y)
    return plotted

In [ ]:
def SaveFigure(fig,fileName):
    plotType="Project_Algorithms/Tracking_Algorithms/Tracked_ParcelTimeDistributions_Plotting"
    plotFileName = f"{fileName}_{ModelData.res}_{ModelData.t_res}_{ModelData.Np_str}{suffix}.jpg"
    plottingFileName = GetPlottingDirectory(plotFileName, plotType)
    print(f"Saving figure to {plottingFileName}")
    fig.savefig(plottingFileName, dpi=300, bbox_inches='tight')
    # plt.close(fig)

In [ ]:
#############################################
#CALCULATION

In [ ]:
#dictionary setup
parcelTypes=['CL','nonCL','SBF','ColdPool']; parcelDepths=['SHALLOW','DEEP']
initialTimesDictionary = SetupInitialTimesDictionary(parcelTypes,parcelDepths)
distributionDictionary = SetupDistributionDictionary(parcelTypes,parcelDepths)

#calculation
timeseriesDictionary = CalculateTimeSeriesDictionary(trackedArrays)
distributionDictionary = CalculateDistributions(initialTimesDictionary,distributionDictionary)
depthProbabilityDictionary = ComputeDepthProbabilityDictionary(distributionDictionary)

In [ ]:
#############################################
#PLOTTING
#y=binned count by initial time

In [ ]:
fig = MakeParcelTimeSeriesFigure(timeseriesDictionary, normalize=False)
SaveFigure(fig, fileName="1. TimeSeries_absolute")

In [ ]:
fig=MakeParcelTimeSeriesFigure(timeseriesDictionary,normalize=True) #y/=max(y)
# SaveFigure(fig, fileName="1. TimeSeries")

In [ ]:
#############################################
#OTHER PLOTTING

In [ ]:
def PlotDistributions(dictionary,
                      hours = ModelData.dt/3600,
                      probabilityType="time",
                      units="%"):
    
    fig, axes = plt.subplots(3, 3, figsize=(15, 10), sharex=True, sharey=True)

    variableKeys = ['left_right', 'left', 'right']
    parcelTypes = ['CL', 'nonCL', 'SBF']

    colors = {'SHALLOW': 'green', 'DEEP': 'blue'}
    linestyles = {'CL': '-', 'nonCL': '--', 'SBF': '-.'}
    
    yLabelDict = {"parcelDepth_time": "P(parcelDepth | T = t)",
                  "time_parcelDepth": "P(T = t | parcelDepth)"}
    yLabel = yLabelDict.get(probabilityType, "")

    for i, parcelType in enumerate(parcelTypes):
        for j, variableKey in enumerate(variableKeys):

            ax = axes[i, j]

            for parcelDepth in dictionary[parcelType]:

                y = dictionary[parcelType][parcelDepth][variableKey].copy()                
                if probabilityType == "time_parcelDepth": y = y / np.sum(y); 
                y[y==0] = np.nan
                ax.plot(
                    timeAxis, y*100,
                    color=colors[parcelDepth],
                    linestyle=linestyles[parcelType]
                )
                ax.set_xlim(left=10)

            # --- Titles ---
            if i == 0:
                ax.set_title(variableKey, fontsize=14)

            if j == 0:
                
                secondLabel = "P(parcelDepth | t)"
                if i == 1:
                    ax.set_ylabel(f"{yLabel} ({units})\n{parcelType}", fontsize=15)
                else:
                    ax.set_ylabel(f"{parcelType}", fontsize=15)

            if i == 2:
                ax.set_xlabel("Time (hrs)",fontsize=12)

            # ax.set_ylim(bottom=0)
            ax.tick_params(axis='both', labelsize=14)

    for ax in axes.flat:
        ax.set_ylim(bottom=0)
    plt.tight_layout(rect=[0, 0, 1, 0.92])

 
    # --- Legends ---
    # TrackedProfiles_Plotting_CLASS.AddCategoryLegend(fig,parcelTypes=parcelTypes,bbox=(0.5, 1.0))
    TrackedProfiles_Plotting_CLASS.AddDepthLegend(axes[0,1])

    return fig

def PlotAreDeepMoreCommonOnLeftOrRight(depthProbabilityDictionary):
    
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    
    parcelTypes = ['CL', 'nonCL', 'SBF']
    for i, parcelType in enumerate(parcelTypes):
    
        ax = axes[i]
    
        # --- DEEP ---
        deep_right = depthProbabilityDictionary[parcelType]['DEEP']['right'].copy()
        deep_left  = depthProbabilityDictionary[parcelType]['DEEP']['left'].copy()
        deep_right[deep_right==0] = np.nan
        deep_left[deep_left==0] = np.nan
        diff_deep = deep_right - deep_left
        
        
    
        # --- SHALLOW ---
        shallow_right = depthProbabilityDictionary[parcelType]['SHALLOW']['right'].copy()
        shallow_left  = depthProbabilityDictionary[parcelType]['SHALLOW']['left'].copy()
        shallow_right[shallow_right==0] = np.nan
        shallow_left[shallow_left==0] = np.nan
        diff_shallow = shallow_right - shallow_left
    
        # --- plot ---
        ax.plot(timeAxis,diff_deep*100, color='blue', label='DEEP')
        ax.plot(timeAxis,diff_shallow*100, color='green', label='SHALLOW')
    
        ax.axhline(0, color='k', linestyle='--')
    
        ax.set_title(parcelType,fontsize=14)
        ax.set_xlabel("Time (hrs)",fontsize=12)
    
        if i == 0:
            ax.set_ylabel("Δ [ P(parcelDepth | Right of SBF, T = t)\n- P(parcelDepth | Left of SBF, T = t)] ",fontsize=12)
    
        ax.legend()

        ax.set_xlim(left=10)
        ax.tick_params(axis='both', labelsize=14)

    return fig

def PlotDeepProbabilityDifference_byParcelType(distributionDictionary,
                                               parcelTypes=['CL','nonCL']):    
    
    keys = ['left_right', 'left', 'right']
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    
    for i, key in enumerate(keys):
    
        ax = axes[i]
    
        # --- CL ---
        CL_deep = distributionDictionary[parcelTypes[0]]['DEEP'][key]
        CL_shallow = distributionDictionary[parcelTypes[0]]['SHALLOW'][key]
        total_CL = CL_deep + CL_shallow
    
        p_deep_CL = np.zeros_like(CL_deep, dtype=float)
        mask = total_CL > 0
        p_deep_CL[mask] = CL_deep[mask] / total_CL[mask]
    
        # --- nonCL ---
        nCL_deep = distributionDictionary[parcelTypes[1]]['DEEP'][key]
        nCL_shallow = distributionDictionary[parcelTypes[1]]['SHALLOW'][key]
        total_nCL = nCL_deep + nCL_shallow
    
        p_deep_nCL = np.zeros_like(nCL_deep, dtype=float)
        mask = total_nCL > 0
        p_deep_nCL[mask] = nCL_deep[mask] / total_nCL[mask]
    
        # --- difference ---
        diff = p_deep_CL - p_deep_nCL
    
        ax.plot(timeAxis,diff*100,color='blue')
    
        ax.axhline(0, linestyle='--', color='black')
        ax.set_title(key, fontsize=14)
        ax.set_xlabel("Time (hrs)",fontsize=12)
    
        if i == 0:
            ax.set_ylabel(f"Δ = P(DEEP|{parcelTypes[0]}) - P(DEEP|{parcelTypes[1]})",fontsize=14)
    
        ax.set_xlim(left=10)
        ax.tick_params(axis='both', labelsize=14)
    
    plt.tight_layout()

    return fig

def PlotDeepProbabilityDifference_byParcelType(distributionDictionary,
                                               parcelTypes=['CL','nonCL']):    
    
    keys = ['left_right', 'left', 'right']
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    
    for i, key in enumerate(keys):
    
        ax = axes[i]
    
        # --- CL ---
        CL_deep = distributionDictionary[parcelTypes[0]]['DEEP'][key]
        CL_shallow = distributionDictionary[parcelTypes[0]]['SHALLOW'][key]
        total_CL = CL_deep + CL_shallow
    
        p_deep_CL = np.zeros_like(CL_deep, dtype=float)
        p_shallow_CL = np.zeros_like(CL_shallow, dtype=float)
    
        mask_CL = total_CL > 0
        p_deep_CL[mask_CL] = CL_deep[mask_CL] / total_CL[mask_CL]
        p_shallow_CL[mask_CL] = CL_shallow[mask_CL] / total_CL[mask_CL]
    
        # --- nonCL ---
        nCL_deep = distributionDictionary[parcelTypes[1]]['DEEP'][key]
        nCL_shallow = distributionDictionary[parcelTypes[1]]['SHALLOW'][key]
        total_nCL = nCL_deep + nCL_shallow
    
        p_deep_nCL = np.zeros_like(nCL_deep, dtype=float)
        p_shallow_nCL = np.zeros_like(nCL_shallow, dtype=float)
    
        mask_nCL = total_nCL > 0
        p_deep_nCL[mask_nCL] = nCL_deep[mask_nCL] / total_nCL[mask_nCL]
        p_shallow_nCL[mask_nCL] = nCL_shallow[mask_nCL] / total_nCL[mask_nCL]
    
        # --- differences ---
        diff_deep = p_deep_CL - p_deep_nCL
        diff_shallow = p_shallow_CL - p_shallow_nCL

        # --- mask ---
        min_count = 10  # or 5 depending on your data
        total_samples = (
            CL_deep + CL_shallow +
            nCL_deep + nCL_shallow
        )
        valid_mask = total_samples > min_count
        diff_deep[~valid_mask] = np.nan
        diff_shallow[~valid_mask] = np.nan
    
        # --- plot ---
        ax.plot(timeAxis, diff_deep * 100, color='blue', label='DEEP')
        # ax.plot(timeAxis, diff_shallow * 100, color='green', label='SHALLOW')
    
        ax.axhline(0, linestyle='--', color='black')
        ax.set_title(key, fontsize=14)
        ax.set_xlabel("Time (hrs)", fontsize=12)
    
        if i == 0:
            ax.set_ylabel(f"Δ = P(parcelDepth|{parcelTypes[0]})\n- P(parcelDepth|{parcelTypes[1]})", fontsize=14)
    
        ax.set_xlim(left=10)
        ax.tick_params(axis='both', labelsize=14)
    
        ax.legend()

    plt.tight_layout()

    return fig

timeAxis=(np.arange(ModelData.Ntime)*ModelData.dt/3600)+6

In [ ]:
# fig=PlotDistributions(depthProbabilityDictionary,probabilityType="parcelDepth_time") #y_deep/(y_shallow+y_deep)
# SaveFigure(fig, fileName="2. P(parcelDepth | T = t)")
# #==> it is more likely for a parcel to be shallow than deep at all times

In [ ]:
# fig = PlotAreDeepMoreCommonOnLeftOrRight(depthProbabilityDictionary) #y_deep_right/(y_shallow_right+y_deep_right) - y_deep_left/(y_shallow_left+y_deep_left)
# SaveFigure(fig, fileName="3. P(parcelDepth | T = t) Right vs Left")
# #==> shallow (deep) clouds occur more (less) often right of SBF in the morning, while deep (shallow) occur more (less) often right of SBF in the afternoon

In [ ]:
# fig=PlotDistributions(distributionDictionary,probabilityType="time_parcelDepth") #y/=sum(y)
# SaveFigure(fig, fileName="4. P(T = t | parcelDepth)")
# # Deep parcels are more likely to occur later in time, while shallow parcels are more likely earlier.

In [ ]:
# #y_deep_CL/(y_shallow_CL+y_deep_CL) - y_deep_nonCL/(y_shallow_nonCL+y_deep_nonCL)
# fig = PlotDeepProbabilityDifference_byParcelType(distributionDictionary,
#                                                  parcelTypes=['CL','nonCL']) 
# SaveFigure(fig, fileName="5. P(DEEP | CL)vs nonCL")

# # CL parcels are consistently more likely to develop into deep convection than nonCL parcels, especially right of SBF. However, left of the sea-breeze boundary, the difference between CL and nonCL is highly variable and lacks a consistent signal.

# # fig = PlotDeepProbabilityDifference_byParcelType(distributionDictionary,
# #                                                  parcelTypes=['SBF','ColdPool'])
# # #SBF are more conducive to deep convection than ColdPools